# parents-dict-by-argidx — ex1: build parents dict — skip non-Tensors, keep original argidx

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `parents-dict-by-argidx`. Running the final beacon cell reports progress against the `Backprop: Parents dict by argidx` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parents dict by argidx` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parents-dict-by-argidx`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parents-dict-by-argidx"
DD_SUBTOPIC = "Backprop: Parents dict by argidx"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Parents dict by argidx — quick refresher

`Recipe.parents` maps **arg position → the input Tensor at that position**, skipping any non-Tensor inputs (ints, floats, shape tuples, ...):

```python
parents = {idx: a for idx, a in enumerate(args) if isinstance(a, Tensor)}
```

Two rules:
- **Skip non-Tensors.** A `multiply(t, 3.0)` call must produce   `parents == {0: t}`, NOT `{0: t, 1: 3.0}` — gradients only flow through   Tensors. The reverse pass would crash trying to add a float to a Tensor   grad otherwise.
- **Keep the original argnum.** The reverse pass uses the dict key to look   up the matching back fn: `BACK_FUNCS.get(func, argnum)`. Renumbering   (e.g. building a list of present Tensors) would break this lookup.

### Exercise 1 — build parents dict — skip non-Tensors, keep original argidx

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the dict-comprehension parents-builder pattern: filter out non-Tensor inputs while preserving the original positional index as the dict key.
> Keywords: parents, argidx, dict-comprehension, filter-non-tensor
> ```

**KCs targeted:** `parents-dict-by-argidx`, `unbox-args-tensor-to-array`

Implement `build_parents(args)`. Given a tuple of positional inputs (some `MiniTensor`, some plain Python scalars / shape tuples / anything else), return a dict mapping **the original argidx** to **the MiniTensor at that position**:

```
build_parents((t1, 3.0, t2))         == {0: t1, 2: t2}
build_parents((5, t1, (1, 2), t2))   == {1: t1, 3: t2}
build_parents((1.0, 2.0))            == {}
```

Two rules — both critical:

**1. Skip non-Tensors.** Use `isinstance(a, MiniTensor)`. If a `multiply(t, 3.0)` call leaks the float 3.0 into `parents`, the reverse pass later tries to add a float to a Tensor grad and crashes — the wrong side of the type system.

**2. Keep the ORIGINAL argidx as the key.** Do NOT collapse `(t1, 3.0, t2)` to `{0: t1, 1: t2}` — the second entry must be `2`, not `1`, because the back-fn lookup is by `(forward_fn, argnum)` with the ORIGINAL argnum. If you renumber, `BACK_FUNCS.get_back_func(func, 1)` returns the wrong back fn at reverse time.

The canonical one-liner is `{idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)}`. Write it (or any equivalent loop).

In [ ]:
class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad


def build_parents(args: tuple) -> dict:
    """Return {argidx: MiniTensor} for each MiniTensor in args, in original order."""
    raise NotImplementedError()


def _test_ex1():
    # --- empty / all-non-Tensor inputs ---
    assert build_parents(()) == {}, 'empty tuple should give empty dict'
    assert build_parents((1, 2.0, 'x')) == {}, 'no Tensors -> empty dict'

    # --- single Tensor at argnum=0 ---
    t1 = MiniTensor(t.tensor([1.0]))
    assert build_parents((t1,)) == {0: t1}

    # --- multiple Tensors, contiguous ---
    t2 = MiniTensor(t.tensor([2.0]))
    p = build_parents((t1, t2))
    assert p == {0: t1, 1: t2}, f'two-tensor case: {p}'

    # --- Tensor in arg-0 only, float at arg-1 (multiply x by scalar) ---
    p = build_parents((t1, 3.0))
    assert p == {0: t1}, f'multiply(t, 3.0): {p}'

    # --- float at arg-0, Tensor at arg-1 — argnum must stay 1, NOT collapse to 0 ---
    p = build_parents((3.0, t1))
    assert p == {1: t1}, (
        f'arg-1 Tensor must keep argnum=1, got {p} '
        f'(renumbering would break BACK_FUNCS dispatch)'
    )

    # --- mixed: int, Tensor, tuple, Tensor ---
    p = build_parents((5, t1, (1, 2, 3), t2))
    assert p == {1: t1, 3: t2}, f'mixed: {p}'

    # --- Tensors at non-consecutive positions ---
    t3 = MiniTensor(t.tensor([3.0]))
    p = build_parents((t1, 'sep', t2, 7, t3))
    assert p == {0: t1, 2: t2, 4: t3}, f'non-consecutive: {p}'

    # --- identity preserved: dict values must BE the same objects ---
    vals = list(p.values())
    assert vals[0] is t1, 'dict value must be the same object as input'
    assert vals[1] is t2, 'dict value must be the same object as input'
    assert vals[2] is t3, 'dict value must be the same object as input'

    # --- raw torch.Tensors should be SKIPPED (only MiniTensors count as parents) ---
    raw = t.tensor([1.0])
    p = build_parents((raw, t1))
    assert p == {1: t1}, (
        f'raw torch.Tensor should be skipped (only MiniTensor counts), got {p}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def build_parents(args: tuple) -> dict:
    return {
        idx: a
        for idx, a in enumerate(args)
        if isinstance(a, MiniTensor)
    }
```

**`enumerate` before `if`.** Order matters in the comprehension: we first attach the original index to each arg via `enumerate`, THEN filter. Doing it the other way (filtering then enumerating the survivors) would re-number — exactly the bug rule 2 warns against.

**Why `isinstance(a, MiniTensor)` and not `hasattr(a, 'array')`.** Duck-typing on `.array` would catch random objects that happen to have an `.array` attribute — e.g. a `numpy.ndarray` literally has an `.array` interface protocol. `isinstance` is precise: we want *the wrapper class*, not anything array-shaped.

**The dual of this is `unbox`.** Where `build_parents` keeps the MiniTensors (filtered, keyed by argnum), `unbox_args` does the opposite — replaces each MiniTensor with its `.array` for the forward call, leaves non-Tensors alone. Same `isinstance` check, different transform.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()